# Batch API 입력 준비

비동기 제출 전에 JSONL 요청을 완성하고 로컬에서 검증합니다.

## 실행 환경 준비

## 코드 읽기와 개념 경계

JSONL은 파일 전체가 하나의 JSON 배열인 형식이 아니라 각 줄이 독립된 JSON 객체인 형식입니다. 각 Batch 요청에는 결과를 원래 업무와 연결할 `custom_id`, HTTP 메서드, 엔드포인트, 요청 본문이 들어갑니다. 예제는 세 객체를 줄 단위로 직렬화하고 다시 읽어 필수 키와 식별자 중복을 검사합니다. 여기까지는 로컬 입력 검증이며 업로드나 배치 생성은 실행하지 않습니다.

Batch는 즉시 응답이 필요한 대화보다 야간 분류·대량 평가처럼 완료를 기다릴 수 있는 작업에 맞습니다. 제출 순서와 결과 순서가 같다고 가정하지 말고 `custom_id`로 결합해야 합니다. 일부 요청만 실패할 수 있으므로 배치 전체 상태와 각 줄의 오류를 나누어 재처리합니다.

In [1]:
from pathlib import Path
from dotenv import load_dotenv

here = Path.cwd().resolve()
chapter_root = next((p for p in (here, *here.parents) if p.name == "ch34-cost-rate-limits" and (p / "notebooks").is_dir()), None)
if chapter_root is None:
    raise FileNotFoundError("ch34-cost-rate-limits 실습 폴더 안에서 실행하세요.")
env_path = next((p / ".env" for p in (here, *here.parents) if (p / ".env").exists()), None)
if env_path:
    load_dotenv(env_path)
print("Python 준비 완료", "API 키 파일 확인" if env_path else "API 키 파일 없음")

Python 준비 완료 API 키 파일 확인


## 세 요청을 JSONL로 만들기

각 줄은 고유한 `custom_id`, 메서드, URL, 요청 본문을 가집니다.

In [2]:
import json

items = ["배송 지연", "환불 요청", "상품 문의"]
rows = [{"custom_id": f"ticket-{i}", "method": "POST", "url": "/v1/responses",
         "body": {"model": "gpt-5-mini", "input": f"다음 문의를 분류하세요: {text}"}}
        for i, text in enumerate(items, 1)]
batch_path = chapter_root / "runs/batch-input.jsonl"
batch_path.parent.mkdir(parents=True, exist_ok=True)
batch_path.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in rows) + "\n")
print(batch_path, "요청 수:", len(rows))

[사용자 폴더]/working/claude/projects/한국경제신문/labs/part7/ch34-cost-rate-limits/runs/batch-input.jsonl 요청 수: 3


## 로컬 구조 검증

파일 업로드와 배치 생성은 비용이 들고 외부 비동기 작업을 시작하므로 이 실습에서는 제출하지 않습니다.

In [3]:
loaded = [json.loads(line) for line in batch_path.read_text().splitlines()]
required = {"custom_id", "method", "url", "body"}
valid = all(required <= row.keys() and row["url"] == "/v1/responses" for row in loaded)
print("JSONL 구조 유효:", valid)
assert valid and len({row["custom_id"] for row in loaded}) == len(loaded)

JSONL 구조 유효: True


## 실패 재현

한 줄을 일반 JSON 배열로 감싸면 JSONL의 ‘한 줄에 한 객체’ 규칙을 어깁니다.

In [4]:
bad_text = json.dumps(rows, ensure_ascii=False)
bad_lines = [json.loads(line) for line in bad_text.splitlines()]
print("첫 줄 타입:", type(bad_lines[0]).__name__)
print("요청 객체로 판정:", all(isinstance(x, dict) for x in bad_lines))

첫 줄 타입: list
요청 객체로 판정: False


## 수정

각 요청 객체를 독립된 줄에 직렬화하고 다시 읽어 확인합니다.

In [5]:
good_lines = [json.loads(line) for line in batch_path.read_text().splitlines()]
print("각 줄이 요청 객체:", all(isinstance(x, dict) for x in good_lines))
print("실제 Batch 제출: 수행하지 않음")
assert len(good_lines) == 3

각 줄이 요청 객체: True
실제 Batch 제출: 수행하지 않음
